In [1]:
# We will produce root files containing:
# 1. Michels
# 2. Throughgoing muons

# Runs over the extracted BeamClusterAnalysis ntuples

# For an authoritative set of selection cuts, see Steven Doran's thesis (doc 6712)


print('Loading Packages...\n')
%matplotlib osx
import sys, os    
import re
import uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import trange
import numpy as np
from scipy.optimize import curve_fit
import pandas as pd
font = {'family' : 'serif', 'size' : 14 }
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm' # Set the math font to Computer Modern
mpl.rcParams['legend.fontsize'] = 16
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
# Set default figure and axes face colors to white
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print('done')

Loading Packages...

done


In [2]:
''' Selection criteria for throughgoing muons
    - Only 1 track in the MRD (signaling its a single muon)
    - TankMRDCoinc + Veto coincidence, telling us it originated upstream the detector and passed all the way through
    - Brighest cluster
    - Very high energy cluster (1000 < cluster pe < 6000) -- this can be restricted further (we will not enforce this from the get go)
    - charge distribution is focused, with almost every PMT seeing some level of light (charge balance < 0.2) (also will not enforce this right away)
    - beam-correlated, cluster time within spill window
    - charge barycenter downstream (z > 0)
    - "throughgoing" track, in which all layers of the MRD are hit
    - greater than 100 PMT hits (enforce the muon does not clip the tank)
'''

def throughgoing(T1,TG1,TMRD1,NV1,B1,CCB1,CH1,CT,hitZ,hitPE):
    if(T1==0):    # 1 track (1st because this is the strictest selection criteria)
        return False
    if(TG1==0):   # Throughgoing (hits all layers of the MRD)
        return False
    if(TMRD1==0): # TankMRDCoinc
        return False
    if(NV1==1):   # NoVeto
        return False
    if(B1==0):    # Brightest
        return False
    if(CCB1>0.2 or CCB1<0): # Cluster Charge Balance < 0.2
        return False
    if(CH1<100):   # indicative the muon topology is not center-aligned
        return False
    if(CT>1750 or CT<200):  # inside spill window
        return False
    a = 0
    for i in range(len(hitZ)):  # charge barycenter downstream
        a += hitZ[i]*hitPE[i]
    if(a<0):
        return False
    
    return True


df = pd.read_csv('FullTankPMTGeometry.csv')
downstream = []; pmt_type = []
for i in range(len(df['channel_num'])):
    downstream.append(df['z_pos'][i]-1.681)   # transform geometric center to the center of the water tank
    if df['PMT_type'][i] == 'LUX':
        pmt_type.append(1)
    elif df['PMT_type'][i] == 'ETEL':
        pmt_type.append(2)
    elif df['PMT_type'][i] == 'Hamamatsu':
        pmt_type.append(3)
    elif df['PMT_type'][i] == 'Watchboy':
        pmt_type.append(4)
    elif df['PMT_type'][i] == 'Watchman':
        pmt_type.append(5)


cluster_time = []; cluster_pe = []; cluster_qb = []; cluster_hits = []; hitID = []; hitPE = []; hitT = []; hitPMTID = []; hitY = []
downstream_charge = []; upstream_charge = [];

# adjust accordingly
directory = 'EBV2_May_2025_data_v1/'
file_names = [file for file in os.listdir(directory) if os.path.isfile(os.path.join(directory, file))]
print('There are: ', len(file_names), ' files\n')

# POT file - can just grab one of the 'POT tables' under Steven Doran's author page on docdb
# A combined file can be found here: /pnfs/annie/persistent/users/doran/datasets/POT_EBV2_finalized.csv
csv_file = 'POT_EBV2_May_2025_data_v1.csv'
pot_df = pd.read_csv(csv_file)
pot_df["run"] = pot_df["FILE"].str.extract(r"_(\d+)\.root").astype(int)
pot_dict = {
    run: (row['E_TOR875[e12]'], row['E_TOR860[e12]']) 
    for run, row in pot_df.set_index("run").iterrows()
}

counter = 0; total_POT_TOR875 = 0; total_POT_TOR860 = 0
for file_name in file_names:
    
    # for testing, use a small % of the files
    #if counter > 0:
    #    continue
        
    print('\nRun: ', file_name, '(', (counter), '/', len(file_names), ')')
    print('------------------------------------------------------------')

    # extract run number from the root files
    match = re.search(r'R(\d+)', file_name)
    if not match:
        print(f"Skipping {file_name}: could not extract run number")
        continue
    run_number = int(match.group(1))

    # lookup run number in POT dictionary
    if run_number not in pot_dict:
        print(f"Skipping {file_name}: run {run_number} not found in POT csv file")
        continue

    E_TOR875, E_TOR860 = pot_dict[run_number]
    print(f"E_TOR875 = {E_TOR875}")
    print(f"E_TOR860 = {E_TOR860}")
    total_POT_TOR875 += E_TOR875
    total_POT_TOR860 += E_TOR860
    
    with uproot.open(os.path.join(directory, file_name)) as file:
    
        Event = file["data"]
        CT2 = Event["cluster_time_BRF"].array()     # BRF-corrected timing
        CPE1 = Event["cluster_PE"].array()
        CCB1 = Event["cluster_Qb"].array()
        NOC1 = Event["number_of_clusters"].array()
        CH1 = Event["cluster_Hits"].array()
        B1 = Event["isBrightest"].array()
        TMRD1 = Event["TankMRDCoinc"].array()
        MRD_yes = Event["MRD_activity"].array()
        T1 = Event["MRD_Track"].array()
        NV1 = Event["NoVeto"].array()
        HZ1 = Event['hitZ'].array()
        HPE1 = Event['hitPE'].array()
        HT1 = Event['hitT'].array()
        HY1 = Event['hitY'].array()
        HID1 = Event['hitID'].array()
        TG1 = Event['MRDThrough'].array()
        
        counter += 1

        events_per_run = 0
        for i in trange(len(CT2)):

            is_thru = throughgoing(T1[i],TG1[i],TMRD1[i],NV1[i],B1[i],CCB1[i],CH1[i],CT2[i],HZ1[i],HPE1[i])
            if(is_thru==False):
                continue
            
            cluster_time.append(CT2[i])
            cluster_pe.append(CPE1[i])
            cluster_hits.append(CH1[i])
            cluster_qb.append(CCB1[i])
            hitID.append(HID1[i])
            hitPE.append(HPE1[i])
            hitT.append(HT1[i])
            hitY.append(HY1[i])
            
            b = []; c = 0; d = 0;
            for j in range(len(HID1[i])):
                indy = int(HID1[i][j]) - 332
                
                # upstream / downstream
                if downstream[indy] > 0:
                    c += HPE1[i][j]
                else:
                    d += HPE1[i][j]
                    
                b.append(pmt_type[indy])
                
            hitPMTID.append(b)
            downstream_charge.append(c)
            upstream_charge.append(d)
            
            events_per_run += 1

        print(events_per_run, ' events')
        
        
total_POT_TOR875 = total_POT_TOR875*(1e12)/(1e20)
total_POT_TOR860 = total_POT_TOR860*(1e12)/(1e20)

print('\nAfter selection cuts, we have: ', len(cluster_time), ' thru-going muon candidates\n\n')
print('Total POT (e20)')
print('---------------')
print('E_TOR875: ' + str(total_POT_TOR875))
print('E_TOR875: ' + str(total_POT_TOR860))

There are:  132  files


Run:  R4083_extracted_data.ntuple.root ( 0 / 132 )
------------------------------------------------------------
E_TOR875 = 4465412.3838442825
E_TOR860 = 4342725.070503506


100%|█████████████████████████████████| 100647/100647 [00:19<00:00, 5242.30it/s]


206  events

Run:  R4431_extracted_data.ntuple.root ( 1 / 132 )
------------------------------------------------------------
E_TOR875 = 521295.8301163244
E_TOR860 = 517557.930903426


100%|███████████████████████████████████| 36466/36466 [00:06<00:00, 5841.36it/s]


24  events

Run:  R4404_extracted_data.ntuple.root ( 2 / 132 )
------------------------------------------------------------
E_TOR875 = 1858213.5098333324
E_TOR860 = 1820929.3398621215


...

15  events

Run:  R4090_extracted_data.ntuple.root ( 130 / 132 )
------------------------------------------------------------
E_TOR875 = 5322059.346070191
E_TOR860 = 5294703.756111941


100%|█████████████████████████████████████| 1925/1925 [00:00<00:00, 5052.85it/s]


5  events

Run:  R4422_extracted_data.ntuple.root ( 131 / 132 )
------------------------------------------------------------
E_TOR875 = 2395066.7920349743
E_TOR860 = 2376385.141429679


100%|█████████████████████████████████| 109351/109351 [00:19<00:00, 5672.87it/s]

102  events

After selection cuts, we have:  6154  thru-going muon candidates


Total POT (e20)
---------------
E_TOR875: 1.786431575546526
E_TOR875: 1.7669955123984948


In [3]:
# chain hitT values to minimum hitT
# so when you look at hitT, 0 = min hit time within a cluster

hitT_adj = []
for i in trange(len(hitT)):
    array = []; mini = min(hitT[i])
    for j in range(len(hitT[i])):
        array.append(hitT[i][j] - mini)
    hitT_adj.append(array)
        
print('done')

100%|██████████████████████████████████████| 6154/6154 [00:09<00:00, 617.98it/s]

done


In [88]:
print('\nWriting to root tree...')
root_file = uproot.create('throughgoing_muons.root')

tree_data = {
    "cluster_time": cluster_time,
    "cluster_pe": cluster_pe,
    "cluster_Qb": cluster_qb,
    "cluster_hits": cluster_hits,
    "downstream_pe": downstream_charge,
    "upsteam_pe": upstream_charge,
    "hitT": hitT_adj,
    "hitPE": hitPE,
    "hitID": hitID,
    "hitPMTID": hitPMTID
}

root_file["Muons"] = tree_data
root_file.close()

print('\ndone\n')


Writing to root tree...

done



In [48]:
# BRF-corrected spill structure for the throughgoing muons

time_adj = []
for i in range(len(cluster_time)):
    if cluster_pe[i] > 2000:
        time_adj.append(cluster_time[i])


bins = np.arange(200,1752,2)
        
fig, ax = plt.subplots(figsize=(12, 4))
plt.hist(cluster_time, bins = bins, histtype = 'stepfilled', color = 'royalblue', alpha = 0.75)
plt.hist(cluster_time, bins = bins, linewidth = 2, histtype = 'step', color = 'k')
plt.xlim([150,1800])
plt.ylabel('events / 2ns', fontsize = 16)
plt.xlabel('cluster time [ns]', fontsize = 16, loc = 'right')
ax.xaxis.set_minor_locator(MultipleLocator(25))
ax.yaxis.set_minor_locator(MultipleLocator(1))
plt.title('Throughgoing muon timing distribution', fontsize = 16)
ax.text(250, 24, f'{1.77}e20 POT', color='black', weight = 'bold', fontsize=14)
ax.text(1545, 27.75, 'ANNIE', color='blue', weight = 'bold', fontsize=10)
ax.text(1650, 27.75, 'Preliminary', color='black', fontsize=10)
#plt.savefig('plots/Thoughgoing_muon_timing_plot_all_events.png',
#            dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()


In [117]:
''' Selection criteria for Michels
    1. Select mother dirt muon
        - Veto activity, but no MRD (or Tank/MRD coincidence)
        - Must be a secondary cluster (candidate for a Michel)
        - Must have an extended readout (so we don't bias our Michel sample towards earlier times within the 2us prompt window)
        - Must be the brightest cluster in the prompt window
        - Have at least 100 PMT hits
        - Large cluster charge, but upper limit to account for potential MRD inefficiencies (1000 < cluster PE < 4000) --> 4000 will limit muons from even reaching the MRD
        - Isotropic charge distribution (charge balance < 0.2) --> removes weird events with a ton of charge in one corner of the tank
        - Muon candidate is within spill window, correlated with beam activity
        - charge barycenter downstream (it is traveling in the beam direction) 

    2. Select Michel candidate
        - Must be between 200ns and 5us after the dirt muon (lifetime is ~2us, cutoff at 5us to avoid afterpulsing effects that take place at 6us)
        - total charge less than 800 pe
        - (NOT INCLUDED) at least 20 PMT hits --> helps centralize the vertices
        - charge balance < 0.2 to avoid highly concentrated charge that could be due to other particle types or noise effects
'''


def dirt(MRD_yes,TMRD1,NV1,NOC1,EXT,B1,CH1,CPE1,CCB1,CT,hitZ,hitPE):
    if(MRD_yes==1):  # any MRD activity
        return False
    if(TMRD1==1):    # TankMRDCoinc
        return False
    if(NV1==1):      # NoVeto
        return False
    if(NOC1==1):     # can't be the only cluster
        return False
    if(EXT==0):      # must have extended window to look for michels
        return False
    if(B1==0):       # Brightest cluster
        return False
    if(CH1<100):      # at least 100 PMT hits
        return False
    if(CPE1<1000 or CPE1>4000):  # 1000 < cluster PE < 4000
        return False
    if(CCB1>0.2 or CCB1<0):      # Cluster Charge Balance < 0.2
        return False
    if(CT>1750 or CT<200):        # inside spill window
        return False
    a = 0
    for i in range(len(hitZ)):   # charge barycenter downstream
        a += hitZ[i]*hitPE[i]
    if(a<0):
        return False
    return True

def Michel(CT2,CPE1,CH1,CCB1):
    if(CT2>5000 or CT2<200):     # at least 200ns after prior cluster, and no more than 5us after
        return False
    if(CPE1<=0 or CPE1>=800):    # total charge less than 800pe
        return False
    #if(CH1<20):                  # at least 20 PMT hits
    #    return False
    if(CCB1>=0.2 or CCB1<=0):    # CCB < 0.2
        return False
    return True


df = pd.read_csv('FullTankPMTGeometry.csv')
downstream = []; pmt_type = []
for i in range(len(df['channel_num'])):
    downstream.append(df['z_pos'][i]-1.681)   # transform geometric center to the center of the water tank
    if df['PMT_type'][i] == 'LUX':
        pmt_type.append(1)
    elif df['PMT_type'][i] == 'ETEL':
        pmt_type.append(2)
    elif df['PMT_type'][i] == 'Hamamatsu':
        pmt_type.append(3)
    elif df['PMT_type'][i] == 'Watchboy':
        pmt_type.append(4)
    elif df['PMT_type'][i] == 'Watchman':
        pmt_type.append(5)


cluster_time = []; cluster_pe = []; cluster_qb = []; cluster_hits = []; hitID = []; hitPE = []; hitT = []; hitPMTID = [];

directory = 'EBV2_May_2025_data_v1/'
file_names = [file for file in os.listdir(directory) if os.path.isfile(os.path.join(directory, file))]
print('There are: ', len(file_names), ' files\n')

# POT file
csv_file = 'POT_EBV2_May_2025_data_v1.csv'
pot_df = pd.read_csv(csv_file)
pot_df["run"] = pot_df["FILE"].str.extract(r"_(\d+)\.root").astype(int)
pot_dict = {
    run: (row['E_TOR875[e12]'], row['E_TOR860[e12]']) 
    for run, row in pot_df.set_index("run").iterrows()
}

counter = 0; total_POT_TOR875 = 0; total_POT_TOR860 = 0
for file_name in file_names:
    
    #if counter > 10:
    #    continue
        
    print('\nRun: ', file_name, '(', (counter), '/', len(file_names), ')')
    print('------------------------------------------------------------')

    # extract run number from the root files
    match = re.search(r'R(\d+)', file_name)
    if not match:
        print(f"Skipping {file_name}: could not extract run number")
        continue
    run_number = int(match.group(1))

    # lookup run number in POT dictionary
    if run_number not in pot_dict:
        print(f"Skipping {file_name}: run {run_number} not found in POT csv file")
        continue

    E_TOR875, E_TOR860 = pot_dict[run_number]
    print(f"E_TOR875 = {E_TOR875}")
    print(f"E_TOR860 = {E_TOR860}")
    total_POT_TOR875 += E_TOR875
    total_POT_TOR860 += E_TOR860
    
    with uproot.open(os.path.join(directory, file_name)) as file:
    
        Event = file["data"]
        CT2 = Event["cluster_time_BRF"].array()     # BRF-corrected timing
        CPE1 = Event["cluster_PE"].array()
        CCB1 = Event["cluster_Qb"].array()
        CN1 = Event["cluster_Number"].array()
        NOC1 = Event["number_of_clusters"].array()
        CH1 = Event["cluster_Hits"].array()
        B1 = Event["isBrightest"].array()
        TMRD1 = Event["TankMRDCoinc"].array()
        MRD_yes = Event["MRD_activity"].array()
        EXT = Event["hadExtended"].array()
        NV1 = Event["NoVeto"].array()
        HZ1 = Event['hitZ'].array()
        HPE1 = Event['hitPE'].array()
        HT1 = Event['hitT'].array()
        HID1 = Event['hitID'].array()
        
        counter += 1

        michels_per_run = 0
        for i in trange(len(CT2)):

            is_dirt = dirt(MRD_yes[i],TMRD1[i],NV1[i],NOC1[i],EXT[i],B1[i],CH1[i],CPE1[i],CCB1[i],CT2[i],HZ1[i],HPE1[i])
            if(is_dirt==False):
                continue
                
            # now look for michel candidates
            is_michel = False
            for k in range(i+1, len(CT2)):
                if CN1[k] != 0:     # not a new event
                    # keep all secondary clusters as candidates
                    adj_time = CT2[k] - CT2[i]
                    is_michel = Michel(adj_time,CPE1[k],CH1[k],CCB1[k])
                    if(is_michel==False):
                        continue
                    # michel candidate parameters
                    cluster_time.append(adj_time)   
                    cluster_pe.append(CPE1[k])
                    cluster_qb.append(CCB1[k])
                    cluster_hits.append(CH1[k])
                    hitID.append(HID1[k])
                    hitPE.append(HPE1[k])
                    hitT.append(HT1[k])
                    
                    b = [];
                    for j in range(len(HID1[k])):
                        indy = int(HID1[k][j]) - 332
                        b.append(pmt_type[indy])

                    hitPMTID.append(b)
                    
                    michels_per_run += 1
                    
                else:
                    break

        print(michels_per_run, ' events')
        
        
total_POT_TOR875 = total_POT_TOR875*(1e12)/(1e20)
total_POT_TOR860 = total_POT_TOR860*(1e12)/(1e20)

print('\nAfter selection cuts, we have: ', len(cluster_time), ' dirt muon candidates\n\n')
print('Total POT (e20)')
print('---------------')
print('E_TOR875: ' + str(total_POT_TOR875))
print('E_TOR860: ' + str(total_POT_TOR860))

There are:  132  files


Run:  R4083_extracted_data.ntuple.root ( 0 / 132 )
------------------------------------------------------------
E_TOR875 = 4465412.3838442825
E_TOR860 = 4342725.070503506


100%|█████████████████████████████████| 100647/100647 [00:22<00:00, 4502.11it/s]


90  events

Run:  R4431_extracted_data.ntuple.root ( 1 / 132 )
------------------------------------------------------------
E_TOR875 = 521295.8301163244
E_TOR860 = 517557.930903426


100%|███████████████████████████████████| 36466/36466 [00:07<00:00, 4817.14it/s]


14  events

Run:  R4404_extracted_data.ntuple.root ( 2 / 132 )
------------------------------------------------------------
E_TOR875 = 1858213.5098333324
E_TOR860 = 1820929.3398621215


...

50  events

Run:  R4090_extracted_data.ntuple.root ( 130 / 132 )
------------------------------------------------------------
E_TOR875 = 5322059.346070191
E_TOR860 = 5294703.756111941


100%|█████████████████████████████████████| 1925/1925 [00:00<00:00, 4833.19it/s]


0  events

Run:  R4422_extracted_data.ntuple.root ( 131 / 132 )
------------------------------------------------------------
E_TOR875 = 2395066.7920349743
E_TOR860 = 2376385.141429679


100%|█████████████████████████████████| 109351/109351 [00:22<00:00, 4776.14it/s]

66  events

After selection cuts, we have:  3371  dirt muon candidates


Total POT (e20)
---------------
E_TOR875: 1.786431575546526
E_TOR875: 1.7669955123984948


In [127]:
# chain hitT values to minimum hitT

hitT_adj = []
for i in trange(len(hitT)):
    array = []; mini = min(hitT[i])
    for j in range(len(hitT[i])):
        array.append(hitT[i][j] - mini)
    hitT_adj.append(array)
        
print('done')

100%|█████████████████████████████████████| 3371/3371 [00:03<00:00, 1023.39it/s]

done


In [128]:
print('\nWriting to root tree...')
root_file = uproot.create('Michels.root')

tree_data = {
    "cluster_time": cluster_time,
    "cluster_pe": cluster_pe,
    "cluster_Qb": cluster_qb,
    "cluster_hits": cluster_hits,
    "hitT": hitT_adj,
    "hitPE": hitPE,
    "hitID": hitID,
    "hitPMTID": hitPMTID
}

root_file["Michels"] = tree_data
root_file.close()

print('\ndone\n')


Writing to root tree...

done

